# 03 · Color and value
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rusparrish/visual-thinking-lens/blob/main/notebooks/03_color_and_value.ipynb)

OKLab lightness and chroma, a deterministic five-color palette, and **where** light, dark and chroma sit in the frame.

In [1]:
# Works locally (pip install -e .) or in Colab.
try:
    import vtl
except ImportError:
    %pip install -q git+https://github.com/rusparrish/visual-thinking-lens
    import vtl
from pathlib import Path
IMAGES = Path("../examples/images") if Path("../examples/images").exists() else None
if IMAGES is None:  # Colab: fetch the sample images
    !git clone -q --depth 1 https://github.com/rusparrish/visual-thinking-lens /content/vtl-repo
    IMAGES = Path("/content/vtl-repo/examples/images")
print("vtl", vtl.__version__, "| images:", IMAGES)

vtl 0.1.0 | images: ../examples/images


In [2]:
c = vtl.measure_color(IMAGES / "still_life.jpg")
{k: (round(v, 3) if isinstance(v, float) else v) for k, v in c.items() if not k.startswith("palette_")}

{'okl_L_mean': 0.422,
 'okl_L_std': 0.105,
 'okl_L_p05': 0.28,
 'okl_L_p95': 0.649,
 'light_mass_fraction': 0.15,
 'dark_mass_fraction': 0.15,
 'high_key_fraction': 0.017,
 'deep_dark_fraction': 0.238,
 'midtone_fraction': 0.745,
 'light_sdi': 0.318,
 'dark_sdi': 0.322,
 'light_delta_x': -0.085,
 'light_delta_y': 0.008,
 'dark_delta_x': 0.172,
 'dark_delta_y': 0.192,
 'spotlight_concentration': 0.177,
 'shadow_concentration': 0.462,
 'okc_C_mean': 0.077,
 'okc_C_std': 0.026,
 'okc_C_p85': 0.105,
 'hue_entropy': 0.467,
 'chroma_entropy': 0.726,
 'color_sdi': 0.403,
 'chroma_delta_x': -0.011,
 'chroma_delta_y': -0.024,
 'chroma_focality': 0.187,
 'lightness_contrast_p95_p05': 0.368,
 'dominant_hue_occupancy': 0.707,
 'effective_palette_count': 4.238,
 'color_version': 'color-v0.4.8'}

In [3]:
from IPython.display import HTML
def swatches(c):
    cells = "".join(
        f'<div style="flex:{c[f"palette_{i}_weight"]:.3f};background:{c[f"palette_{i}_hex"]};height:48px" title="{c[f"palette_{i}_hex"]}"></div>'
        for i in range(1, 6))
    return HTML(f'<div style="display:flex;width:420px;border-radius:4px;overflow:hidden">{cells}</div>')
swatches(c)

Widths are palette weights. `effective_palette_count` (1/Σw²) says how many colors are really doing the work.

### Value structure across the samples

In [4]:
import pandas as pd
rows = [{"image": p.stem, **vtl.measure_color(p)} for p in sorted(IMAGES.glob("*.jpg"))]
cols = ["image", "okl_L_mean", "lightness_contrast_p95_p05", "light_delta_y", "dark_delta_y",
        "spotlight_concentration", "hue_entropy", "effective_palette_count"]
pd.DataFrame(rows)[cols].round(3)

,image,okl_L_mean,lightness_contrast_p95_p05,light_delta_y,dark_delta_y,spotlight_concentration,hue_entropy,effective_palette_count
0,abstract,0.894,0.041,-0.293,0.245,0.451,0.275,1.531
1,collapse,0.417,0.541,-0.269,0.163,0.684,0.254,3.931
2,drawing,0.626,0.496,-0.194,0.083,0.758,0.122,4.811
3,frog,0.525,0.443,-0.048,0.045,0.242,0.390,3.651
4,landscape,0.427,0.640,-0.255,0.071,0.872,0.494,4.636
5,portrait,0.353,0.659,-0.097,0.250,0.536,0.388,3.693
6,room,0.412,0.492,-0.109,0.283,0.314,0.297,3.688
7,still_life,0.422,0.368,0.008,0.192,0.177,0.467,4.238
8,study,0.505,0.304,-0.114,-0.037,0.278,0.248,4.585
9,walking,0.366,0.476,-0.334,0.104,0.856,0.271,4.710


- `light_delta_y < dark_delta_y`: light sits above dark (sky over ground, lit face over shadowed body).
- `spotlight_concentration`: how much of the lightest 15% is one connected region.
- `hue_entropy` ignores near-neutral pixels (chroma < 0.02).